# Inverse Design: Multi-Objective Physics-ML Optimization

## 1. Abstract
This notebook demonstrates the **Inverse Design** engine. The core logic uses an optimizer (Optuna) to minimize a loss function that balances ML predictions with strict physical constraints. This allows us to search the astronomical chemical space for target performance metrics ($E_g$, $PCE$, $T_{80}$).

## 2. Mathematical Framework: Bayesian Search
The search objective $J$ is defined as the error of the prediction $f_{ML}(x)$ relative to the user target $T$, penalized by the **Physical Manifold** criteria:

$$
J(x) = |f_{ML}(x) - T| + P(x)
$$

Where $P(x)$ is a **Physics Penalty** defined as:

$$
P(x) = 
\begin{cases} 
0 & \text{if } 0.82 < t(x) < 1.05 \\
10^6 & \text{otherwise}
\end{cases}
$$

### 2.1. The Sampler Logic: Tree-structured Parzen Estimator (TPE)
Optuna uses Bayesian optimization to model the probability of a composition $x$ given its score $J$. It maintains two distributions:
*   $l(x)$: Density of 'good' trials.
*   $g(x)$: Density of 'bad' trials.

The algorithm suggests new compositions that maximize the **Expected Improvement (EI)**:

$$
EI(x) \propto \frac{l(x)}{g(x)}
$$

By maximizing this ratio, the model 'learns' which chemical combinations ($Cs$ vs $FA$, $I$ vs $Br$) yield stable, high-efficiency devices without exhaustive grid searching.

In [1]:
import pandas as pd
import numpy as np
import optuna
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"

# Mock optimization data to visualize the Sampler's behavior
trials = []
for i in range(200):
    t = np.random.uniform(0.7, 1.2)
    error = abs(1.55 - (1.2 + 0.4*t)) + (1e6 if t < 0.8 or t > 1.05 else 0)
    trials.append({'trial': i, 'tolerance_factor': t, 'objective': error})

df_trials = pd.DataFrame(trials)
print(f"Simulated {len(df_trials)} optimization trials.")

Simulated 200 optimization trials.


## 3. Visualizing Sampler Focus
Notice how the objective function 'pushes' the trials away from the forbidden regions (High Objective values) and clusters them within the stable window. The sampler uses historical trials to 'prune' regions of the chemical space that violate crystallographic rules.

In [ ]:
fig = px.scatter(
    df_trials[df_trials['objective'] < 100], 
    x='trial', 
    y='tolerance_factor', 
    color='objective', 
    title="Sampler Trajectory: Exploration vs. Physical Constraint",
    labels={'tolerance_factor': 'Tolerance Factor (t)', 'trial': 'Optimization Step'}
)
fig.add_hrect(y0=0.8, y1=1.05, fillcolor="green", opacity=0.1, annotation_text="Physically Allowed")
fig.update_layout(width=900, height=500, xaxis_title="Optimization Step", yaxis_title="Structural Descriptor (t)")
fig.show()

## 4. Scientific Limitations and Search Constraints

### 4.1. The 'Black-Hole' Problem
If the ML model has a systematic bias (e.g., it over-predicts $T_{80}$ for a specific unphysical composition), the optimizer will get 'trapped' in that region. This is why strict physical penalties ($t, \mu$) are mandatory to keep the search grounded.

### 4.2. Local Optima
Perovskite stability is a multi-modal landscape. The TPE sampler might converge on a 'Standard' stable composition (e.g., MAPI) and fail to explore radical new configurations (e.g., low-dimensional mixtures) unless the exploration parameter is carefully tuned.

### 4.3. Synthesis Feasibility
The optimizer suggests a stoichiometry (e.g., $A_{0.17}B_{0.83}...$). However, it does not account for whether such a mixture is **Synthetically Accessible** (e.g., solubility limits of precursors or phase segregation during cooling).